In [1]:
!git clone https://github.com/CryAndRRich/codapath.git

Cloning into 'codapath'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 61 (delta 34), reused 37 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 1.36 MiB | 15.50 MiB/s, done.
Resolving deltas: 100% (34/34), done.


In [2]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

/kaggle/working/codapath


In [3]:
!pip install -r requirements.txt
!pip install -U huggingface_hub hf-transfer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 14.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 107.0 MB/s eta 0:00:0000:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1


In [4]:
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login

login("HUGGINGFACE_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Đang tải vinid/plip...")
snapshot_download(repo_id="vinid/plip")

print("Đang tải PubMedBERT...")
snapshot_download(repo_id="microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext")

print("Đang tải BiomedCLIP...")
snapshot_download(repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")

Đang tải vinid/plip...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Đang tải PubMedBERT...


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Đang tải BiomedCLIP...


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

'/root/.cache/huggingface/hub/models--microsoft--BiomedCLIP-PubMedBERT_256-vit_base_patch16_224/snapshots/9f341de24bfb00180f1b847274256e9b65a3a32e'

In [5]:
import sys
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [6]:
import yaml
import torch

In [7]:
from run import main

In [8]:
PATHMNIST_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/pathmnist_224.npz"
HISTOSET_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/HistoSet-5x14/HistoSet-5x14"
SKINTISSUE_PATH = "/kaggle/input/datasets/cryandrrich/nckh2026/SkinTissue/SkinTissue/tiles"

DATA_DICT = {
    "pathmnist": PATHMNIST_PATH,
    "histoset": HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH
}

In [9]:
CONFIG_PATH = "config/config.yaml"

# pathmnist, histoset, skintissue
DATASET = "pathmnist"

# random, coreset, codapath
# entropy, margin, badge, typiclust, activeft
SAMPLER_NAME = "codapath"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

hyper = config.get("hyperparameters", {})
dataset_info = config["datasets"][DATASET]

In [10]:
main(
    data_path=DATA_DICT[DATASET],
    sampler_name=SAMPLER_NAME,
    num_classes=dataset_info["num_classes"],
    cumulative_budget=config["cumulative_budget"],
    data_descriptions=dataset_info["descriptions"],
    prompt_templates=config["prompt_templates"],
    rank_lora=hyper["rank_lora"],
    num_epochs=hyper["num_epochs"],
    learn_rate=hyper["learning_rate"],
    alpha=hyper["alpha"],
    device=torch.device(config["device"]),
    random_seed=config["random_seed"],
    save_dir=f"checkpoints/{DATASET}",
    verbose=True
)

Device: cuda
Random seed: 42
Train size: 22400 | Test size: 5600


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: vinid/plip
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
vision_model.embeddings.class_embedding                        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
vision_model.pre_layrnorm.bias                         

...

Kích hoạt torch.compile()


W0411 14:06:52.709000 55 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch [01/25] | Loss: 9.0779
Epoch [02/25] | Loss: 8.8156
Epoch [03/25] | Loss: 8.5712
Epoch [04/25] | Loss: 8.2238
Epoch [05/25] | Loss: 8.0385
Epoch [06/25] | Loss: 7.7260
Epoch [07/25] | Loss: 7.6624
Epoch [08/25] | Loss: 7.4820
Epoch [09/25] | Loss: 7.2383
Epoch [10/25] | Loss: 7.0605
Epoch [11/25] | Loss: 7.0341
Epoch [12/25] | Loss: 6.8772
Epoch [13/25] | Loss: 6.7626
Epoch [14/25] | Loss: 6.5965
Epoch [15/25] | Loss: 6.5298
Epoch [16/25] | Loss: 6.4447
Epoch [17/25] | Loss: 6.3909
Epoch [18/25] | Loss: 6.2522
Epoch [19/25] | Loss: 6.1864
Epoch [20/25] | Loss: 6.0938
Epoch [21/25] | Loss: 6.0032
Epoch [22/25] | Loss: 5.9628
Epoch [23/25] | Loss: 5.8742
Epoch [24/25] | Loss: 5.7773
Epoch [25/25] | Loss: 5.7823


Accuracy : 79.27%
Precision: 79.70%
Recall   : 79.03%
Macro F1 : 78.61%


Saved model parameters to: checkpoints/histoset/codapath_budget_50.pth
